In [0]:
from pyspark.sql import functions as F
print("Gold layer starting")

In [0]:
df = spark.table('silver.lung_cancer')

gold_country = df.groupBy('country') \
    .agg(
        F.count('*').alias('total_patients'),
        F.sum(F.when(F.col('lung_cancer_diagnosis') == 'Yes', 1).otherwise(0)).alias('cancer_cases'),
        F.avg('age').alias('avg_age'),
        F.sum(F.when(F.col('smoker') == 'Yes', 1).otherwise(0)).alias('smokers'),
        F.avg('cigarettes_per_day').alias('avg_cigarettes_per_day'),
        F.avg('mortality_rate').alias('avg_mortality_rate')
    ) \
    .withColumn('cancer_rate', F.round(F.col('cancer_cases') / F.col('total_patients'), 4)) \
    .withColumn('smoker_rate', F.round(F.col('smokers') / F.col('total_patients'), 4))

display(gold_country)

In [0]:
gold_country.write.format('delta').mode('overwrite') \
    .saveAsTable('gold.risk_by_country')

print(f"Saved {gold_country.count()} countries to gold.risk_by_country")

In [0]:
gold_features = spark.table('silver.lung_cancer').select(
    'age', 'gender', 'country', 'smoker',
    'years_of_smoking', 'cigarettes_per_day',
    'passive_smoker', 'family_history',
    'air_pollution_exposure', 'occupational_exposure',
    'indoor_pollution', 'lung_cancer_diagnosis'
).dropna(subset=['lung_cancer_diagnosis', 'age'])

gold_features.write.format('delta').mode('overwrite') \
    .saveAsTable('gold.ml_features')

print(f"Saved {gold_features.count():,} rows to gold.ml_features")

In [0]:
spark.sql('OPTIMIZE gold.risk_by_country ZORDER BY (country)')
spark.sql('OPTIMIZE gold.ml_features ZORDER BY (country, age)')

spark.sql('VACUUM gold.risk_by_country RETAIN 168 HOURS')
spark.sql('VACUUM gold.ml_features RETAIN 168 HOURS')

print('Optimization complete')
display(spark.sql('DESCRIBE HISTORY gold.risk_by_country'))

In [0]:
df_who = spark.table('silver.who_pollution')

gold_who = df_who.groupBy('Country') \
    .agg(
        F.avg('ValueNumeric').alias('avg_deaths'),
        F.max('ValueNumeric').alias('max_deaths'),
        F.min('Year').alias('earliest_year'),
        F.max('Year').alias('latest_year')
    ) \
    .withColumnRenamed('Country', 'country')

gold_who.write.format('delta').mode('overwrite') \
    .saveAsTable('gold.who_country_summary')

print(f"Saved {gold_who.count()} countries to gold.who_country_summary")

In [ ]:
df_ab = spark.table('silver.abstracts')

gold_pubmed = df_ab \
    .withColumn('keyword', F.explode(F.col('risk_keywords'))) \
    .groupBy('keyword') \
    .agg(F.count('*').alias('total_mentions')) \
    .orderBy(F.col('total_mentions').desc())

gold_pubmed.write.format('delta').mode('overwrite') \
    .saveAsTable('gold.pubmed_keyword_summary')

print(f"Saved {gold_pubmed.count()} keywords to gold.pubmed_keyword_summary")
display(gold_pubmed)

In [0]:
df_aq = spark.table('silver.air_quality')

gold_aq = df_aq.select(
    'country', 'avg_pm2_5', 'avg_pm10',
    'avg_aqi', 'max_aqi', 'aqi_category'
)

gold_aq.write.format('delta').mode('overwrite') \
    .saveAsTable('gold.air_quality_summary')

print(f"Saved {gold_aq.count()} countries to gold.air_quality_summary")
display(gold_aq)

In [0]:
spark.sql('OPTIMIZE gold.who_country_summary ZORDER BY (country)')
spark.sql('OPTIMIZE gold.air_quality_summary ZORDER BY (country)')
spark.sql('OPTIMIZE gold.pubmed_keyword_summary ZORDER BY (keyword)')

print('All Gold tables optimized')

# Final check — list all Gold tables
display(spark.sql('SHOW TABLES IN gold'))